In [1]:
class Player:   #1.1-1.2
    def __init__(self,id,name,hp):
        try:
            self._id=int(id)
            self._hp=max(0,int(hp))
            self._name=name.strip().title()
        except ValueError:
            print('Введите соответствующую данные')
            self._id, self._hp, self._name = 0, 0, "Unknown"

    def __str__(self):
        return f"Player(id={self._id}, name='{self._name}', hp={self._hp})"
    def __del__(self):
        print(f"Player {self._name} удалён")
    @classmethod
    def from_string(cls,data):
        parts=[x.strip() for x in data.split(',')]
        if len(parts)!=3:
            raise ValueError(f"Неверный формат строки: ожидалось 3 значения, получено {len(parts)}")
        return cls(*parts)




p = Player.from_string("2, alice , 90")
print(p)

Player(id=2, name='Alice', hp=90)


In [2]:
class Item:          #1.3
    def __init__(self,id, name, power):
        self.id=id
        self.name=name.strip()
        self.power=power
    def __str__(self):
        return f"Item(id={self.id}, name='{self.name}', power={self.power})"
    def __eq__(self,other):
        if not isinstance(other,Item):
            return False
        return self.id==other.id
    def __hash__(self):
        return hash(self.id)
i=Item(1, "Sword", 50)
i1=Item(1,'Axe',30)
class Inventory:           #1.4
    def __init__(self):
        self.items=[]
    def add_item(self,item):
        for x in self.items:
            if x.id==item.id:
                return
        self.items.append(item)
    def remove_item(self,item_id):
        for x in self.items:
            if x.id==item_id:
                self.items.remove(x)
                break
    def get_items(self):
        return self.items
    def unique_items(self):
        return set(self.items)
    def to_dict(self):
        res = {x.id: x for x in self.items}
        return res
    def get_strong_items(self,min_power):      #1.5
        return [x for x in filter(lambda y:y.power>=min_power,self.items)]

inv=Inventory()
inv.add_item(i)
for x,y in inv.to_dict().items():
    print(x,y)
for x in inv.get_strong_items(10):
    print(x)

1 Item(id=1, name='Sword', power=50)
Item(id=1, name='Sword', power=50)


In [3]:
from datetime import datetime         #1.6,7
import ast
class Event:
    def __init__(self,type,data):
        allowed={'ATTACK','HEAL','LOOT'}
        if type not in allowed:
            raise ValueError('Invalid event type')
        self.type=type
        self.data=data
        self.timestamp=datetime.now()
event = Event("ATTACK", {"damage": 20})
class Player:
    def __init__(self,name,hp,inventory,id):
        self.name=name
        self.hp=hp
        self.inventory=inventory
        self.id=id
    def handle_event(self,event):
        if event.type=='ATTACK':
            self.hp-=event.data['damage']
        elif event.type=='HEAL':
            self.hp+=event.data['hp']
        elif event.type=='LOOT':
            self.inventory.append(event.data['item'])
class Warrior(Player):
    def handle_event(self,event):
        if event.type=='ATTACK':
            self.hp-=event.data['damage']*0.9
        else:
            super().handle_event(event)
class Mage(Player):
    def handle_event(self,event):
        if event.type=='LOOT':
            item=event.data['item']
            item.power*=1.1
            self.inventory.append(item)
        else:
            super().handle_event(event)
event_attack = Event("ATTACK", {"damage": 100})
player = Warrior('Yamal', 1000, [],4)
player.handle_event(event_attack)
'''print(f"HP после атаки: {yamal.hp}")'''
class Logger:                 #1.8-1.9
    def log(self, event: Event, player: Player, filename: str):
        line = f"{event.timestamp};{player.id};{event.type};{event.data}\n"
        with open(filename, "a",encoding='utf-8') as fn:
            fn.write(line)
    def read_logs(self,filename):
        events=[]
        with open(filename,'r') as fn:
            for line in fn:
                parts=line.strip().split(';')
                if len(parts)<4: continue
                event_type = parts[2]
                data = ast.literal_eval(parts[3])
                event=Event(event_type,data)
                events.append(event)
        return events
logger = Logger()
logger.log(event_attack, player, "game.txt")
events=logger.read_logs("game.txt")
for x in events:
    print(x.type,x.data)


ATTACK {'damage': 100}
ATTACK {'damage': 100}


In [4]:
class EventIterator:       #1.10
    def __init__(self,events):
        self.events=events
        self.index=0
    def __iter__(self):
        return self
    def __next__(self):
        if self.index<len(self.events):
            event=self.events[self.index]
            self.index+=1
            return event
        else:
            raise StopIteration
i=EventIterator(events)
for x in i:
    print(x.type,x.data)

ATTACK {'damage': 100}
ATTACK {'damage': 100}


In [5]:
from random import choice     #1.11-1.12
from random import randint
class Generator:
    def damage_stream(self,events):
        for x in events:
            if x.type=="ATTACK":
                data=x.data
                yield data['damage']

    def generate_events(self,players: list[Player], items: list[Item], n: int):
        events=["ATTACK","HEAL","LOOT"]
        result=[]
        for player in players:
            for _ in range(n):
                event=(lambda:choice(events))()
                if event=="ATTACK":
                    res=Event("ATTACK",{"damage":randint(1,100),"player":player.name})
                elif event=="HEAL":
                    res=Event("HEAL",{"hp":randint(1,100),"player":player.name})
                else:
                    res=Event("LOOT",{'item':choice(items),'player':player.name})
                result.append(res)
        return result
g=Generator()
events= [Event("ATTACK", {"damage": 20}),Event("ATTACK", {"damage": 10}),Event("HEAL", {"hp": 20})]
Naruto=Player("Naruto",1000,[],1)
Sasuke=Player("Sasuke",950,[],2)
players=[Naruto,Sasuke]
items=[Item(1, "Sword", 50),Item(2,'Axe',30)]
for x in g.generate_events(players,items,3):
    if x.type=="LOOT":
        print(x.type,{"item":str(x.data['item'])},x.data['player'])
    else:
        print(x.type,x.data)

ATTACK {'damage': 22, 'player': 'Naruto'}
HEAL {'hp': 48, 'player': 'Naruto'}
LOOT {'item': "Item(id=2, name='Axe', power=30)"} Naruto
LOOT {'item': "Item(id=1, name='Sword', power=50)"} Sasuke
HEAL {'hp': 92, 'player': 'Sasuke'}
LOOT {'item': "Item(id=1, name='Sword', power=50)"} Sasuke


In [6]:
def analyze_logs(event):      #1.13
    total_damage=sum(x.data['damage'] for x in event if x.type=="ATTACK")
    events=[x.type for x in event]
    most_common_event={}
    for x in events:
        if x not in most_common_event:
            most_common_event[x]=0
        most_common_event[x]+=1
    top=[x.data for x in event if x.type=="ATTACK"]
    top_player={}
    for x in top:
        if x['player'] not in top_player:
            top_player[x['player']]=0
        top_player[x['player']]+=x['damage']


    res={"total_damage":total_damage,
         "top_player":sorted(top_player.items(),key=lambda x:-x[1])[0][0],
         "most_common_event":sorted(most_common_event.items(),key=lambda x:-x[1])[0][0]}
    return res
analyze_logs(g.generate_events(players,items,3))

{'total_damage': 65, 'top_player': 'Sasuke', 'most_common_event': 'HEAL'}

In [7]:
decide_action=lambda x:'HEAL' if x.hp<30 else 'LOOT' if len(x.inventory)==0 else 'ATTACK'       #1.14
print(decide_action(Naruto))
print(decide_action(Sasuke))

LOOT
LOOT


In [8]:
class Warrior(Player):     #1.15
    def handle_event(self,event):
        if event.type=='ATTACK':
            self.hp-=event.data['damage']*0.9
        else:
            super().handle_event(event)
class Mage(Player):
    def handle_event(self,event):
        if event.type=='LOOT':
            item=event.data['item']
            item.power*=1.1
            self.inventory.append(item)
        else:
            super().handle_event(event)
event = Event("LOOT", {"item": Item(2,'sword',26)})
event_attack=Event("ATTACK",{"damage":25})
Zoro=Warrior('Zoro',1000,[],4)
Zoro.handle_event(event_attack)
Luffy=Mage('Luffy',1500,[],3)
Luffy.handle_event(event)
Luffy.handle_event(event_attack)
for x in Luffy.inventory:
    print(f'{x},{Luffy.name},{Luffy.id},{Luffy.hp}')
print(f'{Zoro.name},{Zoro.id},{Zoro.inventory},{Zoro.hp}')

Item(id=2, name='sword', power=28.6),Luffy,3,1475
Zoro,4,[],977.5


In [9]:
class Player:         #1.16-1.17
    def __init__(self,name,hp,inventory,id):
        self.name=name
        self.__hp=hp
        self.__inventory=inventory
        self.id=id
    def handle_event(self,event):
        if event.type=='ATTACK':
            self.__hp-=event.data['damage']
        elif event.type=='HEAL':
            self.__hp+=event.data['hp']
        elif event.type=='LOOT':
            self.__inventory.append(event.data['item'])
    @property
    def info(self):
        return f"Персонаж: {self.name} , HP: {self.__hp}"
    def __del__(self):
        print(f"Player {self.name} удалён")

Asta=Player('Asta',750,[],8)
Asta.handle_event(event_attack)
print(Asta.info)
del  Asta

Персонаж: Asta , HP: 725
Player Asta удалён


In [10]:
class Inventory:          #1.18-1.19
    def __init__(self,items):
        self.items=items

    def __iter__(self):
        return iter(self.items)
    def analyze_inventory(self):
        unique_items=set(self.items)
        top_power=max(self.items,key=lambda x:x.power) if self.items else None
        res={
            "unique_items":unique_items,
            "top_power":top_power}
        return res

i=Item(1, "Sword", 50)
i1=Item(2,'Axe',30)
i2=Item(3,'shied',40)
items=[i,i1,i2]
bag=Inventory(items)
for item in bag:
    print(item)
strong_item=[item for item in bag if item.power>=40]
print()
for x in bag.analyze_inventory().values():
    print(x)

Item(id=1, name='Sword', power=50)
Item(id=2, name='Axe', power=30)
Item(id=3, name='shied', power=40)

{<__main__.Item object at 0x0000026D8B619A30>, <__main__.Item object at 0x0000026D8B665260>, <__main__.Item object at 0x0000026D8B665E10>}
Item(id=1, name='Sword', power=50)


In [11]:
import ast            #1.20
import json
def main():
    players=[Player("Naruto",10000,[],1),Player("Sasuke",9000,[],2),
             Player("Sakura",7500,[],3),Player("Kakashi",8000,[],4),
             Warrior("Zoro", 1000, [], 101),Mage("Luffy", 800, [], 102)]
    items=[Item(1, "Sword", 50),Item(2,'Axe',30),
           Item(3,'pickaxe',20),Item(4,'bow',60),
           Item(5,'trident',40)]
    events=["ATTACK","HEAL","LOOT"]
    result=[]
    with open("game_log.txt",'w') as f:
        f.write("")
    print("--- Запуск симуляции ---")
    for player in players:
        for _ in range(2):
            event=choice(events)
            if event=="ATTACK":
                res=Event("ATTACK",{"damage":randint(1,100),"player":player.name})
            elif event=="HEAL":
                res=Event("HEAL",{"hp":randint(1,100),"player":player.name})
            else:
                res=Event("LOOT",{"item":choice(items),"player":player.name})
            player.handle_event(res)
            result.append(res)
    logger=Logger()
    for res in result:
        player_name=res.data['player']
        pl=next(p for p in players if p.name==player_name)
        logger.log(res,pl,"game_log.txt")
    events_from_file=logger.read_logs("game.txt")
    attack_events=[x.data for x in result if x.type=="ATTACK"]
    top_player={}
    most_item_player={}
    loot_events=[(str(x.data['item']),x.data['player']) for x in result if x.type=="LOOT"]
    for x in loot_events:
        if x[1] not in most_item_player:
            most_item_player[x[1]]=[]
        most_item_player[x[1]].append(x[0])
    most_item_player=dict(map(lambda x:(x[0],len(x[1])),most_item_player.items()))
    for x in attack_events:
        if x['player'] not in top_player:
            top_player[x['player']]=0
        top_player[x['player']]+=x['damage']

    print("Игрок с наибольшим уроном:",sorted(top_player.items(),key=lambda x:-x[1])[0][0])
    print("Игрок с максимальным количеством предметов:",sorted(most_item_player.items(),key=lambda x:-x[1])[0][0] if most_item_player else None )
if __name__=="__main__":
    main()
    print("--- Программа завершена ---")

--- Запуск симуляции ---
Игрок с наибольшим уроном: Sasuke
Игрок с максимальным количеством предметов: Zoro
Player Kakashi удалён
Player Sakura удалён
Player Sasuke удалён
Player Naruto удалён
--- Программа завершена ---
